# People Like Me

### Rank other professionals most like me using different methods
#### 1. Standardize Work Experience using O*NET and use cosine similarity
- create embeddings for work experience and  O*NET 
- find similar matches
(O*NET (Occupational Information Network) is the nation's primary source of occupational information, providing a comprehensive, digital database of worker skills, job requirements, and training needs for over 900 occupations. Sponsored by the U.S. Department of Labor, it helps HR professionals, educators, and job seekers analyze jobs and explore careers.)



Setup:  create a .venv -- must be in a workspace to do this.  This .venv will be available to all projects in workspace
Use:  https://chatgpt.com/share/69cd1090-2a98-8328-9f28-b24fa7b362ce

In [ ]:
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("snehaanbhawal/resume-dataset")

# print("Path to dataset files:", path)

#### For HuggingFace, install library "datasets"

In [1]:
# From Hugging Face:
# pip install datasets
# Dataset is located here:  C:\Users\DeloresMincarelli\.cache\huggingface\hub\

import pandas as pd
from datasets import load_dataset


### Import dataset

In [2]:

dataset = load_dataset("med2425/resume-job-fit-merged-v1")

In [3]:
import json

row = dataset["train"][0]
print(json.dumps(row, indent=2, ensure_ascii=False))

{
  "resume": "SummarySelf-motivated accountant  offering a strong work ethic and determination to complete tasks in a timely manner. Accurate and detail-oriented with extensive auditing and finance knowledge.\nHighlightsComplex problem solvingStrong communication skillsExpert in customer relationsPortfolio managementAProficient in Microsoft OfficeMicrosoft Excel expertRisk management expertiseFinancial statement analysisGeneral ledger accounting\nExperienceCurrentto08/2014AccountantApartment Income Reit Corp.–Nashua,NH,Prepare unpaid reports on actual expenses for marketing line of business.Create and maintain pending and process able database.Prepare and setup vendor purchase orders contracts as well as CRX templates.Verify funding and SAP project code against the most recent budget/forecast submission.Key invoices into ePurchase system as well as approve and reconcile invoices.Track invoices from submission to payment on database.Monitor invoice central mailbox that will include inv

In [4]:

train = dataset["train"].to_pandas()
test = dataset["test"].to_pandas()

train["Is_Me"] = 0
test["Is_Me"] = 0

train.head()
# row = train.iloc[0].to_dict()
# print(json.dumps(row, indent=2, ensure_ascii=False))

,resume,jd,label,source,resume_domain,jd_domain,Is_Me
0,SummarySelf-motivated accountant offering a s...,"Our client, a Raleigh-based full-service comme...",Potential Fit,generated_smart,finance,finance,0
1,Professional ProfileSpecialized knowledge of r...,"""All candidates must be directly contracted by...",No Fit,generated_smart,finance,software,0
2,"SummaryEngineering Project Manager\nAmbitious,...",Calling all innovators find your future at Fi...,No Fit,generated_smart,software,software,0
3,SummaryData Protection Consultant with 10 year...,"MUST WORK ON A W2 INDEPENDENTLY, NO SPONSORSHI...",No Fit,generated_smart,software,data,0
4,Career OverviewHighly skilled SOFTWARE QUALITY...,Calling all data nerds who love finance! \nIf ...,No Fit,generated_smart,software,data,0


In [5]:
distinct_resume_domains = sorted(train["resume_domain"].dropna().unique())
distinct_resume_domains

['ai',
 'data',
 'design',
 'engineering',
 'finance',
 'healthcare',
 'hr',
 'legal',
 'management',
 'marketing',
 'sales',
 'software']

In [6]:
# Cleaning  of imported dataset, adding new cols

if "Is_Me" not in train.columns:
    train["Is_Me"] = 0
if "Is_Me" not in test.columns:
    test["Is_Me"] = 0

train_resume = train[["resume", "resume_domain", "source", "Is_Me"]].copy()
test_resume = test[["resume", "resume_domain", "source", "Is_Me"]].copy()

train_resume = train_resume.drop_duplicates(subset=["resume"], keep="first").reset_index(drop=True)
test_resume = test_resume.drop_duplicates(subset=["resume"], keep="first").reset_index(drop=True)

# One ID per distinct resume text (shared across train/test)
combined_resume = pd.concat([train_resume["resume"], test_resume["resume"]], ignore_index=True)
resume_ids, _ = pd.factorize(combined_resume)
n_train = len(train_resume)
train_resume["resume_id"] = resume_ids[:n_train]
test_resume["resume_id"] = resume_ids[n_train:]

train_resume = train_resume[["resume_id", "resume", "resume_domain", "source", "Is_Me"]]
test_resume = test_resume[["resume_id", "resume", "resume_domain", "source", "Is_Me"]]

train_resume.head()

,resume_id,resume,resume_domain,source,Is_Me
0,0,SummarySelf-motivated accountant offering a s...,finance,generated_smart,0
1,1,Professional ProfileSpecialized knowledge of r...,finance,generated_smart,0
2,2,"SummaryEngineering Project Manager\nAmbitious,...",software,generated_smart,0
3,3,SummaryData Protection Consultant with 10 year...,software,generated_smart,0
4,4,Career OverviewHighly skilled SOFTWARE QUALITY...,software,generated_smart,0


### Add your resume from a PDF or `my_resume_extracted.txt`

Install once: `pip install pypdf ipywidgets`

1. Run the next cell (extract helpers).
2. Run the cell after that (optional path + upload widget). If you use **Upload**, pick your PDF, then run the **following** cell again so the widget value is read.
3. Run the last cell: it prefers PDF (upload or `dam_resume.pdf`) if available, otherwise loads `my_resume_extracted.txt`. It removes any prior `source == "me"` rows, then appends your resume to **both** `train` and `test` with `resume_domain='data'`, `source='me'`, `Is_Me=1`. Then **re-run** the Cleaning cell to refresh `train_resume` / `test_resume`.

In [7]:
import io
import re
from pathlib import Path

from pypdf import PdfReader


In [8]:


def organize_resume_text(text: str) -> str:
    """Normalize whitespace and page breaks for cleaner plain text."""
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t\f\v]+", " ", text)
    text = re.sub(r"\n[ \t]+", "\n", text)
    text = re.sub(r"[ \t]+\n", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def extract_text_from_pdf(data: bytes) -> str:
    reader = PdfReader(io.BytesIO(data))
    parts = []
    for page in reader.pages:
        t = page.extract_text()
        if t:
            parts.append(t.strip())
    return organize_resume_text("\n\n".join(parts))


print("Ready: organize_resume_text, extract_text_from_pdf")

Ready: organize_resume_text, extract_text_from_pdf


### Use Upload button to upload pdf resume

In [9]:
import ipywidgets as widgets
from IPython.display import display

NOTEBOOK_DIR = Path.cwd()
# PDF in the same folder as the notebook (when Jupyter cwd is this folder)
MY_RESUME_PDF = NOTEBOOK_DIR / "dam_resume.pdf"

print(f"If no upload button appears (common in Cursor), use a PDF at:\n  {MY_RESUME_PDF.resolve()}\n")

pdf_upload = widgets.FileUpload(accept=".pdf", multiple=False, description="Resume PDF")
display(pdf_upload)

If no upload button appears (common in Cursor), use a PDF at:
  C:\Users\DeloresMincarelli\Documents\Portfolio\peopleLikeMe\dam_resume.pdf



FileUpload(value=(), accept='.pdf', description='Resume PDF')

### Scrape PDF text and save as `my_resume_extracted.txt`

In [10]:
RESUME_TEXT_CACHE = NOTEBOOK_DIR / "my_resume_extracted.txt"


In [ ]:
# extract text from pdf resume

# def _pdf_bytes_from_upload(w) -> bytes | None:
#     if not w.value:
#         return None
#     f0 = w.value[0]
#     content = f0["content"]
#     if hasattr(content, "tobytes"):
#         return content.tobytes()
#     return bytes(memoryview(content))


# pdf_bytes = _pdf_bytes_from_upload(pdf_upload)
# if pdf_bytes is not None:
#     print("Using PDF from upload widget.")
# elif MY_RESUME_PDF.exists():
#     pdf_bytes = MY_RESUME_PDF.read_bytes()
#     print(f"Using PDF from disk: {MY_RESUME_PDF.resolve()}")
# else:
#     raise FileNotFoundError(
#         f"No PDF yet: use the upload widget and re-run this cell, "
#         f"or save your file as:\n  {MY_RESUME_PDF.resolve()}"
#     )

# # here is where text extracted from pdf resume
# my_resume_text = extract_text_from_pdf(pdf_bytes)
# RESUME_TEXT_CACHE.write_text(my_resume_text, encoding="utf-8")

# print(f"Saved text to {RESUME_TEXT_CACHE} ({len(my_resume_text):,} chars).")

Using PDF from upload widget.
Saved text to c:\Users\DeloresMincarelli\Documents\Portfolio\peopleLikeMe\my_resume_extracted.txt (5,427 chars).


In [12]:
# Insert my resume into train_resume and test_resume

my_resume_text = (NOTEBOOK_DIR / "my_resume_extracted.txt").read_text(encoding="utf-8")

# Remove any prior "me" rows so re-running doesn't duplicate
train_resume = train_resume[train_resume["source"] != "me"].reset_index(drop=True)
test_resume = test_resume[test_resume["source"] != "me"].reset_index(drop=True)

next_id = train_resume["resume_id"].max() + 1

new_row = pd.DataFrame(
    [
        {
            "resume_id": next_id,
            "resume": my_resume_text,
            "resume_domain": "data",
            "source": "me",
            "Is_Me": 1,
        }
    ]
)
train_resume = pd.concat([train_resume, new_row], ignore_index=True)
test_resume = pd.concat([test_resume, new_row], ignore_index=True)

print(
    f"Resume text: {len(my_resume_text):,} chars. "
    f"train_resume rows: {len(train_resume)}, test_resume rows: {len(test_resume)}"
)

# Verify
train_resume[train_resume["Is_Me"] == 1]

Resume text: 5,431 chars. train_resume rows: 643, test_resume rows: 478


,resume_id,resume,resume_domain,source,Is_Me
642,642,Delores Mincarelli\n5133166586 ◇ deloresmincar...,data,me,1


### Extract resume section headings

In [13]:
import re

my_resume_df = train_resume[train_resume["Is_Me"] == 1].copy()

if len(my_resume_df) != 1:
    raise ValueError("Expected exactly 1 row where Is_Me == 1")

text = my_resume_df.iloc[0]["resume"]

lines = [line.strip() for line in text.splitlines()]
lines = [line for line in lines if line]

candidate_headings = []

known_headings = {
    "summary",
    "professional summary",
    "experience",
    "work experience",
    "professional experience",
    "education",
    "skills",
    "technical skills",
    "projects",
    "certifications",
    "awards",
}

for line in lines:
    alpha = re.sub(r"[^A-Za-z]", "", line)

    uppercase_ratio = (
        sum(1 for c in alpha if c.isupper()) / len(alpha)
        if len(alpha) > 0 else 0
    )

    is_short = len(line) <= 40
    is_known = line.lower() in known_headings
    looks_upper = line.isupper() or uppercase_ratio > 0.8

    if is_short and (is_known or looks_upper):
        candidate_headings.append(line)

candidate_headings = list(dict.fromkeys(candidate_headings))

print("Candidate headings:")
for h in candidate_headings:
    print("-", h)

Candidate headings:
- SUMMARY
- EXPERIENCE
- SKILLS
- EDUCATION
- CERTIFICATIONS
- PUBLICATIONS


### Standard heading dictionary

In [14]:
heading_map = {
    "SUMMARY": "Summary",
    "EXPERIENCE": "Work Experience",
    "SKILLS": "Skills",
    "EDUCATION": "Education",
    "CERTIFICATIONS": "Certifications",
    "PUBLICATIONS": "Publications",
}

heading_map

{'SUMMARY': 'Summary',
 'EXPERIENCE': 'Work Experience',
 'SKILLS': 'Skills',
 'EDUCATION': 'Education',
 'CERTIFICATIONS': 'Certifications',
 'PUBLICATIONS': 'Publications'}

#### Change resume headings to standard; save as resume_lines_by_section.csv

In [14]:
my_resume_df = train_resume[train_resume["Is_Me"] == 1].copy()

if len(my_resume_df) != 1:
    raise ValueError("Expected exactly 1 row where Is_Me == 1")

resume_id = my_resume_df.iloc[0]["resume_id"]
text = my_resume_df.iloc[0]["resume"]

lines = [line.strip() for line in text.splitlines()]
lines = [line for line in lines if line]

rows = []
current_section = "Unknown"

for line in lines:
    if line in heading_map:
        current_section = heading_map[line]
        continue

    rows.append({
        "resume_id": resume_id,
        "section": current_section,
        "line_text": line
    })

sections_df = pd.DataFrame(rows)

print(sections_df.head(80).to_string(index=False))
sections_df.to_csv("resume_lines_by_section.csv", index=False)
print("Saved resume_lines_by_section.csv")

 resume_id         section                                                                                                                                                                                                                                                     line_text
       642         Unknown                                                                                                                                                                                                                                            Delores Mincarelli
       642         Unknown                                                                                                                                                                                                           5133166586 ◇ deloresmincarelli@yahoo.com ◇ LinkedIn
       642         Summary                                                                                                                                   

### Extract work experience bullets; save as resume_bullets.csv

In [15]:
my_resume_df = train_resume[train_resume["Is_Me"] == 1].copy()

if len(my_resume_df) != 1:
    raise ValueError("Expected exactly 1 row where Is_Me == 1")

resume_id = my_resume_df.iloc[0]["resume_id"]
text = my_resume_df.iloc[0]["resume"]

lines = [line.strip() for line in text.splitlines()]
lines = [line for line in lines if line]

rows = []
current_section = "Unknown"
bullet_id = 1

for line in lines:
    if line in heading_map:
        current_section = heading_map[line]
        continue

    if current_section != "Work Experience":
        continue

    is_bullet = (
        line.startswith("-")
        or line.startswith("•")
        or re.match(r"^\u2022", line) is not None
    )

    if is_bullet:
        bullet_text = re.sub(r"^[\-\•\u2022]\s*", "", line)

        rows.append({
            "resume_id": resume_id,
            "bullet_id": bullet_id,
            "section": current_section,
            "bullet_text": bullet_text,
            "Is_Me": 1
        })
        bullet_id += 1

resume_bullets_df = pd.DataFrame(rows)

print(resume_bullets_df.to_string(index=False))
resume_bullets_df.to_csv("resume_bullets.csv", index=False)
print("Saved resume_bullets.csv")

 resume_id  bullet_id         section                                                                                                                                                                                                                                                  bullet_text  Is_Me
       642          1 Work Experience                                                                                     LLM Fine-Tuning & Evaluation (Healthcare Analytics): Fine Tuned a LLM on laboratory data, and evaluated against baseline LLM, allowing for open discussion on tradeoffs.      1
       642          2 Work Experience Created an automated process using Cursor AI to interpret complex SQL code and craft operational definitions and examples for over 100 Key Performance Indicators.This allows our AI Chat Bot to answer customer questions within the context of the metric.      1
       642          3 Work Experience                              Innovative Problem Solving in Analytics

### Load O*NET data files
https://www.onetcenter.org/database.html#all-files

In [16]:
onet_dir = Path("data/onet")

occupation_df = pd.read_csv(onet_dir / "Occupation Data.txt", sep="\t", dtype=str)
tasks_df = pd.read_csv(onet_dir / "Task Statements.txt", sep="\t", dtype=str)
skills_df = pd.read_csv(onet_dir / "Skills.txt", sep="\t", dtype=str)

print("Occupation Data columns:")
print(occupation_df.columns.tolist())
print()

print("Task Statements columns:")
print(tasks_df.columns.tolist())
print()

print("Skills columns:")
print(skills_df.columns.tolist())

Occupation Data columns:
['O*NET-SOC Code', 'Title', 'Description']

Task Statements columns:
['O*NET-SOC Code', 'Task ID', 'Task', 'Task Type', 'Incumbents Responding', 'Date', 'Domain Source']

Skills columns:
['O*NET-SOC Code', 'Element ID', 'Element Name', 'Scale ID', 'Data Value', 'N', 'Standard Error', 'Lower CI Bound', 'Upper CI Bound', 'Recommend Suppress', 'Not Relevant', 'Date', 'Domain Source']


### Filter O*Net for just the cols we want

In [17]:
# What this is doing:

# occupation_ref_df keeps occupation code, title, description
# tasks_ref_df keeps task phrases you will match against
# skills_ref_df keeps skill phrases you will match against
# filtering Scale ID == 'IM' keeps only skill importance rows and removes the duplicate level rows

# When this finishes, you should have 3 filtered files:

# occupation_ref_df
# onet_tasks_ref.csv
# onet_skills_ref.csv

# Keep only the needed columns
occupation_ref_df = occupation_df[
    ['O*NET-SOC Code', 'Title', 'Description']
].copy()

tasks_ref_df = tasks_df[
    ['O*NET-SOC Code', 'Task ID', 'Task']
].copy()

skills_ref_df = skills_df[
    ['O*NET-SOC Code', 'Element ID', 'Element Name', 'Scale ID', 'Data Value']
].copy()

# Keep only skill importance rows
skills_ref_df = skills_ref_df[
    skills_ref_df['Scale ID'] == 'IM'
].copy()

# Add occupation title
tasks_ref_df = tasks_ref_df.merge(
    occupation_ref_df[['O*NET-SOC Code', 'Title']],
    on='O*NET-SOC Code',
    how='left'
)

skills_ref_df = skills_ref_df.merge(
    occupation_ref_df[['O*NET-SOC Code', 'Title']],
    on='O*NET-SOC Code',
    how='left'
)

# Save
tasks_ref_df.to_csv("onet_tasks_ref.csv", index=False)
skills_ref_df.to_csv("onet_skills_ref.csv", index=False)

print("tasks_ref_df shape:", tasks_ref_df.shape)
print("skills_ref_df shape:", skills_ref_df.shape)

print("\nSample task rows:")
print(tasks_ref_df.head(5).to_string(index=False))

print("\nSample skill rows:")
print(skills_ref_df.head(5).to_string(index=False))

tasks_ref_df shape: (18796, 4)
skills_ref_df shape: (31290, 6)

Sample task rows:
O*NET-SOC Code Task ID                                                                                                                                                                                            Task            Title
    11-1011.00    8823                                                         Direct or coordinate an organization's financial or budget activities to fund operations, maximize investments, or increase efficiency. Chief Executives
    11-1011.00    8824                                                              Confer with board members, organization officials, or staff members to discuss issues, coordinate activities, or resolve problems. Chief Executives
    11-1011.00    8827                                                                                                        Prepare budgets for approval, including those for funding or implementation of programs. Chief E

In [18]:
# check 
print(tasks_ref_df['Task'].isnull().sum())
print(skills_ref_df['Element Name'].isnull().sum())
print(skills_ref_df['Scale ID'].value_counts())

0
0
Scale ID
IM    31290
Name: count, dtype: int64


### Match your resume bullets to O*NET tasks and skills via Embeddings

The idea is simple:

- turn each bullet_text into an embedding
- turn each O*NET task into an embedding
- compare them with cosine similarity
- keep the top matches

### Install sentence-transformers Scikit-learn

sentence-transformers
- A library for turning sentences or short texts into fixed-size vectors (embeddings) so you can compare meaning, not just words.
- Built on PyTorch and Hugging Face Transformers.
Pre-trained models (e.g. MiniLM, MPNet) map text to dense vectors; similar meanings tend to sit close in vector space.
Common uses: semantic search, duplicate detection, clustering, re-ranking, similarity between documents or queries.
It’s the usual choice when you want “what does this sentence mean?” style similarity without training a big model from scratch.

scikit-learn

- The main Python library for classical / tabular machine learning and scientific computing around models.

Together: sentence-transformers gives you text embeddings; scikit-learn can use those vectors (or other features) for clustering, classification, nearest neighbors, etc.

In [19]:
# pip install sentence-transformers scikit-learn

#validate inputs before going further

resume_bullets_df = pd.read_csv("resume_bullets.csv", dtype=str).fillna("")
tasks_ref_df = pd.read_csv("onet_tasks_ref.csv", dtype=str).fillna("")
skills_ref_df = pd.read_csv("onet_skills_ref.csv", dtype=str).fillna("")

print("resume_bullets_df shape:", resume_bullets_df.shape)
print("tasks_ref_df shape:", tasks_ref_df.shape)
print("skills_ref_df shape:", skills_ref_df.shape)

print("\nSample resume bullets:")
print(resume_bullets_df.head(3).to_string(index=False))

print("\nSample tasks:")
print(tasks_ref_df.head(3).to_string(index=False))

print("\nSample skills:")
print(skills_ref_df.head(3).to_string(index=False))


resume_bullets_df shape: (18, 5)
tasks_ref_df shape: (18796, 4)
skills_ref_df shape: (31290, 6)

Sample resume bullets:
resume_id bullet_id         section                                                                                                                                                                                                                                                  bullet_text Is_Me
      642         1 Work Experience                                                                                     LLM Fine-Tuning & Evaluation (Healthcare Analytics): Fine Tuned a LLM on laboratory data, and evaluated against baseline LLM, allowing for open discussion on tradeoffs.     1
      642         2 Work Experience Created an automated process using Cursor AI to interpret complex SQL code and craft operational definitions and examples for over 100 Key Performance Indicators.This allows our AI Chat Bot to answer customer questions within the context of the metric.  

### Convert everything to lists

In [20]:
# convert each row of text into a list
resume_texts = resume_bullets_df["bullet_text"].tolist()
task_texts = tasks_ref_df["Task"].tolist()
skill_texts = skills_ref_df["Element Name"].tolist()

print("Number of resume bullets:", len(resume_texts))
print("Number of tasks:", len(task_texts))
print("Number of skills:", len(skill_texts))

print("\nExample resume text:", resume_texts[0])
print("\nExample task:", task_texts[0])
print("\nExample skill:", skill_texts[0])

Number of resume bullets: 18
Number of tasks: 18796
Number of skills: 31290

Example resume text: LLM Fine-Tuning & Evaluation (Healthcare Analytics): Fine Tuned a LLM on laboratory data, and evaluated against baseline LLM, allowing for open discussion on tradeoffs.

Example task: Direct or coordinate an organization's financial or budget activities to fund operations, maximize investments, or increase efficiency.

Example skill: Reading Comprehension


### Load embedding model
all-MiniLM-L6-v2 (from SentenceTransformers) because it’s:

- Built specifically for semantic similarity
It converts text → vectors (embeddings)
Similar meaning → vectors close together (cosine similarity)
That’s exactly what your workflow needs:
Resume bullets ↔ skills/tasks matching
- Very efficient
Small model (~80MB)
Fast on CPU (no GPU needed)
Great for batching thousands of rows (like O*NET)
- “Good enough” accuracy

Is it free?
- Yes — completely free
Open-source (Apache 2.0 license)
No API calls
Runs locally
No usage limits
No cost per embedding

- That’s why it’s commonly used in: prototypes and offline pipelines

In [21]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "all-MiniLM-L6-v2"

model = SentenceTransformer(MODEL_NAME)

print("Model loaded successfully")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully


### Encode ONLY your resume bullets first

- will create a matrix: 18 rows (one embedding per chunk) × 384 columns (features per embedding).
- in the output of the cell, you will see the first 10 out of 384 embeddings

In [22]:
print("Encoding resume bullets...")

resume_emb = model.encode(resume_texts, show_progress_bar=True)

print("Embedding shape:", resume_emb.shape)
print("First embedding vector (truncated):", resume_emb[0][:10])

Encoding resume bullets...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding shape: (18, 384)
First embedding vector (truncated): [ 0.0318018  -0.00019295 -0.02170281 -0.01921288  0.01609795 -0.04736843
 -0.04208609  0.0728913  -0.06033973 -0.00707478]


### Skills (sample first)

In [28]:
skill_sample = skill_texts[:500]

print("Encoding sample of skills...")

skill_emb_sample = model.encode(skill_sample, show_progress_bar=True)

print("Skill sample embedding shape:", skill_emb_sample.shape)

Encoding sample of skills...


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Skill sample embedding shape: (500, 384)


### Test one bullet for Skills

In [29]:
sim_scores = cosine_similarity(test_bullet_emb, skill_emb_sample)[0]

top_idx = np.argsort(sim_scores)[::-1][:5]

print("\nTop 5 matching skills:")
for idx in top_idx:
    print(f"{sim_scores[idx]:.3f} | {skill_sample[idx]}")


Top 5 matching skills:
0.340 | Monitoring
0.340 | Monitoring
0.340 | Monitoring
0.340 | Monitoring
0.340 | Monitoring


In [ ]:
## Limit the O*NET files for relevant titles, getting too much noise

relevant_titles = [
    "Data",
    "Analyst",
    "Scientist",
    "Database",
    "Statistician",
    "Computer",
    "Software",
    "Information"
]


### Skills test - use subset of skills and test on one bullet

In [31]:
## SKILL TEST ON BULLET: filter skills for relevant titles, getting too much noise
## DE-DUPE BC GETTING SAME SKILL 5X

test_text = "Profiled incoming telemetry data in real time to detect issues and anomalies."

skill_subset_df = skills_ref_df[
    skills_ref_df["Title"].str.contains("|".join(relevant_titles), case=False, na=False)
].copy()

skill_subset_texts = skill_subset_df["Element Name"].tolist()

skill_subset_emb = model.encode(skill_subset_texts, show_progress_bar=True)
test_emb = model.encode([test_text])

sim_scores = cosine_similarity(test_emb, skill_subset_emb)[0]

skill_results_df = skill_subset_df.copy()
skill_results_df["score"] = sim_scores

skill_results_df = (
    skill_results_df
    .sort_values("score", ascending=False)
    .drop_duplicates(subset=["Element Name"])
)

print("Test bullet:")
print(test_text)

print("\nTop 10 unique matching subset skills:")
print(
    skill_results_df[["Element Name", "score", "Title"]]
    .head(10)
    .to_string(index=False)
)

Batches:   0%|          | 0/61 [00:00<?, ?it/s]

Test bullet:
Profiled incoming telemetry data in real time to detect issues and anomalies.

Top 10 unique matching subset skills:
         Element Name    score                                              Title
Operations Monitoring 0.517768 Geoscientists, Except Hydrologists and Geographers
           Monitoring 0.470444                               Computer Programmers
Equipment Maintenance 0.293504                                Database Architects
     Systems Analysis 0.254959                                   Biostatisticians
     Active Listening 0.231609                               Computer Programmers
      Troubleshooting 0.227392                       Operations Research Analysts
  Operations Analysis 0.226617                               Materials Scientists
  Service Orientation 0.218736 Market Research Analysts and Marketing Specialists
   Systems Evaluation 0.207120 Geoscientists, Except Hydrologists and Geographers
      Time Management 0.193135               Compu

### Compare ALL bullets to O*NET Skills

In [32]:
## SKILL EMBEDDING ON ALL BULLETS


from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

# Use your filtered subset
skill_subset_df = skills_ref_df[
    skills_ref_df["Title"].str.contains("|".join(relevant_titles), case=False, na=False)
].copy()

skill_subset_texts = skill_subset_df["Element Name"].tolist()
skill_subset_emb = model.encode(skill_subset_texts, show_progress_bar=True)

results = []

for i, row in resume_bullets_df.iterrows():
    bullet_text = row["bullet_text"]
    bullet_id = row["bullet_id"]
    resume_id = row["resume_id"]

    test_emb = model.encode([bullet_text])
    sim_scores = cosine_similarity(test_emb, skill_subset_emb)[0]

    temp_df = skill_subset_df.copy()
    temp_df["score"] = sim_scores

    temp_df = (
        temp_df
        .sort_values("score", ascending=False)
        .drop_duplicates(subset=["Element Name"])
        .head(5)
    )

    for _, r in temp_df.iterrows():
        results.append({
            "resume_id": resume_id,
            "bullet_id": bullet_id,
            "bullet_text": bullet_text,
            "skill": r["Element Name"],
            "score": r["score"]
        })

skills_match_df = pd.DataFrame(results)

skills_match_df.to_csv("resume_skill_matches.csv", index=False)

print(skills_match_df.head(20).to_string(index=False))

Batches:   0%|          | 0/61 [00:00<?, ?it/s]

resume_id bullet_id                                                                                                                                                                                                                                                  bullet_text                             skill    score
      642         1                                                                                     LLM Fine-Tuning & Evaluation (Healthcare Analytics): Fine Tuned a LLM on laboratory data, and evaluated against baseline LLM, allowing for open discussion on tradeoffs.                        Monitoring 0.340066
      642         1                                                                                     LLM Fine-Tuning & Evaluation (Healthcare Analytics): Fine Tuned a LLM on laboratory data, and evaluated against baseline LLM, allowing for open discussion on tradeoffs.                Systems Evaluation 0.339884
      642         1                                 

In [ ]:
# What you did (clean version)
# 1. You created two “vocabularies”

# From O*NET:

# Skills vocabulary
# short, reusable concepts (e.g., Operations Monitoring, Systems Analysis)
# Tasks vocabulary
# longer activity descriptions (e.g., monitor system operation…)

# 👉 These are your standardized reference concepts

# 2. You embedded both sides

# You turned text into vectors:

# Resume bullets → embeddings
# O*NET skills → embeddings
# O*NET tasks → embeddings

# 👉 Now everything lives in the same vector space

# 3. You compared meaning (not words)

# You used:

# cosine similarity

# to answer:

# “Which O*NET concepts are closest in meaning to this bullet?”

# 4. You selected top matches

# For each bullet:

# top N skills (after deduplication)
# top N tasks (initially)

# 👉 This gives:

# bullet → standardized concepts

# 5. You filtered the search space

# This is a key improvement you made:

# You restricted O*NET to relevant occupations

# Instead of comparing to:

# geothermal operators
# filing clerks
# warehouse inspectors

# You compared to:

# data scientists
# analysts
# software roles

# 👉 This dramatically improved quality


# You are:

# ranking O*NET concepts by semantic similarity

# This matters because:

# results are probabilistic
# you will later apply thresholds, weights, aggregation
# 🧠 What you actually built (this is the big insight)

# You just built:

# A semantic translator from resume language → standardized workforce ontology

# In plain English:

# messy human resume
# → clean, comparable skill signals

### Tasks

In [33]:
print(tasks_ref_df.columns.tolist())
print(tasks_ref_df.head(3).to_string())

['O*NET-SOC Code', 'Task ID', 'Task', 'Title']
  O*NET-SOC Code Task ID                                                                                                                                     Task             Title
0     11-1011.00    8823  Direct or coordinate an organization's financial or budget activities to fund operations, maximize investments, or increase efficiency.  Chief Executives
1     11-1011.00    8824       Confer with board members, organization officials, or staff members to discuss issues, coordinate activities, or resolve problems.  Chief Executives
2     11-1011.00    8827                                                 Prepare budgets for approval, including those for funding or implementation of programs.  Chief Executives


In [35]:
# filter the O*Net tasks to just relevant titles

task_subset_df = tasks_ref_df[
    tasks_ref_df["Title"].str.contains("|".join(relevant_titles), case=False, na=False)
].copy()

print(task_subset_df.shape)
print(task_subset_df[["Title", "Task"]].head(10).to_string(index=False))

(1199, 4)
                                    Title                                                                                                                            Task
Computer and Information Systems Managers Direct daily operations of department, analyzing workflow, establishing priorities, developing standards and setting deadlines.
Computer and Information Systems Managers            Meet with department heads, managers, supervisors, vendors, and others, to solicit cooperation and resolve problems.
Computer and Information Systems Managers                                                                   Review project plans to plan and coordinate project activity.
Computer and Information Systems Managers                                Assign and review the work of systems analysts, programmers, and other computer-related workers.
Computer and Information Systems Managers                                                                     Provide users with technical s

In [39]:
# General cleaning of tasks

task_subset_df = task_subset_df.dropna(subset=["Task"]).copy()
task_subset_df["Task"] = task_subset_df["Task"].astype(str).str.strip()
task_subset_df = task_subset_df[task_subset_df["Task"] != ""].copy()
task_subset_df = task_subset_df.drop_duplicates(subset=["Task"]).copy()

task_subset_texts = task_subset_df["Task"].tolist()
print(task_subset_texts[:5])

['Direct daily operations of department, analyzing workflow, establishing priorities, developing standards and setting deadlines.', 'Meet with department heads, managers, supervisors, vendors, and others, to solicit cooperation and resolve problems.', 'Review project plans to plan and coordinate project activity.', 'Assign and review the work of systems analysts, programmers, and other computer-related workers.', 'Provide users with technical support for computer problems.']


### Embed the tasks

In [40]:
task_subset_emb = model.encode(task_subset_texts, show_progress_bar=True)

Batches:   0%|          | 0/38 [00:00<?, ?it/s]

### Loop through each resume bullet and score against the task embeddings

In [41]:


from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

results = []

for i, row in resume_bullets_df.iterrows():
    bullet_text = row["bullet_text"]
    bullet_id = row["bullet_id"]
    resume_id = row["resume_id"]

    bullet_emb = model.encode([bullet_text])
    sim_scores = cosine_similarity(bullet_emb, task_subset_emb)[0]

    temp_df = task_subset_df.copy()
    temp_df["score"] = sim_scores

    temp_df = (
        temp_df
        .sort_values("score", ascending=False)
        .drop_duplicates(subset=["Task"])
        .head(5)
    )

    for _, r in temp_df.iterrows():
        results.append({
            "resume_id": resume_id,
            "bullet_id": bullet_id,
            "bullet_text": bullet_text,
            "task": r["Task"],
            "score": r["score"]
        })

tasks_match_df = pd.DataFrame(results)

In [42]:
print(tasks_match_df.head(20).to_string(index=False))

resume_id bullet_id                                                                                                                                                                                                                                                  bullet_text                                                                                                                                                         task    score
      642         1                                                                                     LLM Fine-Tuning & Evaluation (Healthcare Analytics): Fine Tuned a LLM on laboratory data, and evaluated against baseline LLM, allowing for open discussion on tradeoffs.                                                                 Evaluate and recommend upgrades or improvements to existing computerized healthcare systems. 0.406847
      642         1                                                                                     LLM Fine-Tuning & 

In [43]:
tasks_match_df.to_csv("resume_task_matches.csv", index=False)

In [44]:
sample_bullet = resume_bullets_df.iloc[0]["bullet_text"]
print(sample_bullet)

tasks_match_df[
    tasks_match_df["bullet_text"] == sample_bullet
].sort_values("score", ascending=False).to_string(index=False)

LLM Fine-Tuning & Evaluation (Healthcare Analytics): Fine Tuned a LLM on laboratory data, and evaluated against baseline LLM, allowing for open discussion on tradeoffs.


'resume_id bullet_id                                                                                                                                                              bullet_text                                                                                                                                                         task    score\n      642         1 LLM Fine-Tuning & Evaluation (Healthcare Analytics): Fine Tuned a LLM on laboratory data, and evaluated against baseline LLM, allowing for open discussion on tradeoffs.                                                                 Evaluate and recommend upgrades or improvements to existing computerized healthcare systems. 0.406847\n      642         1 LLM Fine-Tuning & Evaluation (Healthcare Analytics): Fine Tuned a LLM on laboratory data, and evaluated against baseline LLM, allowing for open discussion on tradeoffs.                             Monitor clinical trials or experiments to ensure adherence to establis